
# Fama–MacBeth Regression

This notebook is a self-contained explanation of **Fama–MacBeth (1973)** regression. 

---



## 1. Cross-Sectional vs Time-Series Regression

### Time-series regression
- One asset
- Many dates

\begin{align}
R_{i,t} &= \alpha_i + \beta_i F_t + \varepsilon_{i,t}
\end{align}

Used to estimate **factor exposures**.

---

### Cross-sectional regression
- Many assets
- One date

\begin{align}
R_{i,t} &= \lambda_{0,t} + \lambda_{1,t} X_{i,t} + u_{i,t}
\end{align}

Used to explain **why assets differ from each other at the same time**.



## 2. Economic Question

> Are assets with higher exposure to a risk factor rewarded with higher expected returns?

Fama–MacBeth provides a clean way to:
- Estimate factor exposures
- Estimate factor risk premia
- Conduct valid inference



## 3. Step 1 — Time-Series Regression (Estimate Betas)

For each asset \(i\), run an OLS regression over time:

\begin{align}
R_{i,t} &= \alpha_i + \beta_i F_t + \varepsilon_{i,t}
\end{align}

- This is standard OLS
- Done **asset by asset**
- Output: one \(\hat{\beta}_i\) per asset

Interpretation:
> “How sensitive is asset \(i\) to the factor?”



## 4. Step 2 — Cross-Sectional Regression (Estimate Risk Prices)

For each time period \(t\), run an OLS regression **across assets**:

\begin{align}
R_{i,t} &= \lambda_{0,t} + \lambda_{1,t} \hat{\beta}_i + u_{i,t}
\end{align}

Key points:
- Betas are **fixed inputs**
- Regression is across assets
- Repeated for every time period

Each regression produces:
\begin{align}
\lambda_{1,t} &= \text{price of risk at time } t
\end{align}

Interpretation:
> “At time \(t\), how much return does the market pay per unit of beta?”



## 5. Step 3 — Average Over Time

You now have a time series:
\begin{align}
\{\lambda_{1,1}, \lambda_{1,2}, \dots, \lambda_{1,T}\}
\end{align}

Compute the average price of risk:

\begin{align}
\bar{\lambda}_1 &= \frac{1}{T}\sum_{t=1}^{T} \lambda_{1,t}
\end{align}

This is the **mean risk premium**.



## 6. Computing the Fama–MacBeth t-statistic

Treat $\lambda_{1,t}$  as a time series.

### Sample variance
\begin{align}
\widehat{\mathrm{Var}}(\lambda_1)
&= \frac{1}{T-1}\sum_{t=1}^{T}(\lambda_{1,t} - \bar{\lambda}_1)^2
\end{align}

### Standard error of the mean
\begin{align}
\mathrm{SE}(\bar{\lambda}_1)
&= \sqrt{\frac{\widehat{\mathrm{Var}}(\lambda_1)}{T}}
\end{align}

### t-statistic
\begin{align}
t &= \frac{\bar{\lambda}_1}{\mathrm{SE}(\bar{\lambda}_1)}
\end{align}



## 7. Hypothesis Test

\begin{align}
H_0 &: \mathbb{E}[\lambda_1] = 0 \\
H_1 &: \mathbb{E}[\lambda_1] \neq 0
\end{align}

Interpretation:
- Null: assets are **not** rewarded for this risk
- Alternative: the factor is **priced**



## 8. Interpretation of the t-statistic

- Large ($|t| > 2$):  
  - Risk premium is consistently positive (or negative)
  - Factor is **priced**

- Small ($|t| < 1$):  
  - Risk premium fluctuates around zero
  - No reliable compensation for risk

Economic meaning:
> “Taking more of this risk systematically pays off.”



## 9. Why This Works

- Betas capture **cross-sectional differences**
- Prices of risk vary **over time**
- Inference uses **time-series variability**, not cross-sectional noise

This avoids:
- Cross-sectional dependence issues
- Overstated t-statistics from pooled OLS



## 10. Important Caveats

- **Autocorrelation in $\lambda_t$**  
  → Use Newey–West standard errors

- **Estimated betas**  
  → Shanken correction

These affect inference, not point estimates.



## 11. One-Sentence Summary

> Fama–MacBeth regression estimates risk premia by averaging cross-sectional prices of risk over time and uses their time-series variability for inference.


# Generalisation to Multiple Factors

Fama–MacBeth regression extends naturally to **multiple risk factors**.

---

### Step 1 — Time-Series Regressions (Estimate Factor Loadings)

For each asset \(i\), run a time-series OLS regression:

$$
R_{i,t}
= \alpha_i
+ \beta_{i1} F_{1,t}
+ \beta_{i2} F_{2,t}
+ \cdots
+ \beta_{iK} F_{K,t}
+ \varepsilon_{i,t}
$$

**Output:** a vector of factor loadings for each asset: $$(\hat{\beta}_{i1}, \hat{\beta}_{i2}, \ldots, \hat{\beta}_{iK})$$

**Interpretation:**  
*“How exposed is asset \(i\) to each risk factor?”*

---

### Step 2 — Cross-Sectional Regressions (Estimate Risk Prices)

For each time period \(t\), run a cross-sectional regression across assets:

$$
R_{i,t}
= \lambda_{0,t}
+ \lambda_{1,t} \hat{\beta}_{i1}
+ \lambda_{2,t} \hat{\beta}_{i2}
+ \cdots
+ \lambda_{K,t} \hat{\beta}_{iK}
+ u_{i,t}
$$

Each period produces:

$$
(\lambda_{1,t}, \lambda_{2,t}, \ldots, \lambda_{K,t})
$$

**Interpretation:**  
*“At time \(t\), how much return does the market pay per unit of each risk?”*

---

### Step 3 — Average Risk Prices Over Time

For each factor \(k\), compute the average price of risk:

$$
\bar{\lambda}_k
= \frac{1}{T}\sum_{t=1}^{T} \lambda_{k,t}
$$

Each $\bar{\lambda}_k$ is the **mean risk premium** for factor \(k\).



---

### Step 4 — Inference

Compute a t-statistic for each factor \(k\):

$$
t_k
= \frac{\bar{\lambda}_k}{\sqrt{\widehat{\mathrm{Var}}(\lambda_k)/T}}
$$

Null hypothesis: $H_0:\ \mathbb{E}[\lambda_k] = 0$

If $|t_k|$ is large, factor \(k\) is **priced**.


# Python Example

In [1]:
import numpy as np
import pandas as pd
from numpy.linalg import lstsq

In [2]:
# Reproducibility
np.random.seed(42)

# Dimensions
T = 120        # time periods (e.g. months)
N = 50         # assets
K = 2          # number of factors

# Simulate factor returns
F = np.random.normal(0, 1, size=(T, K))

# True betas for each asset
true_betas = np.random.normal(0, 1, size=(N, K))

# True risk premia
true_lambda = np.array([0.5, -0.3])

# Simulate asset returns
epsilon = np.random.normal(0, 1, size=(T, N))
R = F @ true_betas.T + epsilon

### Step 1: Time-series regression (estimate betas): $R_{i,t} = \alpha_i + \beta_i^TF_t + \epsilon_{i,t}$

In [3]:
# Add constant for time-series regressions
X_ts = np.column_stack([np.ones(T), F])

# Store estimated betas
beta_hat = np.zeros((N, K))

for i in range(N):
    y = R[:, i]
    coef, _, _, _ = lstsq(X_ts, y, rcond=None)
    beta_hat[i] = coef[1:]  # skip intercept

#### Each row of beta_hat is: $(\hat{\beta}_{i,1} , \hat{\beta}_{i,2} ) $

### Step 2: Cross-sectional regression (estimate lambda t): $R_{i,t} = \lambda_{0,t} + \lambda_t^T \hat{\beta_i} + u_{i, t}$

In [4]:
# Store lambda_t estimates
lambda_t = np.zeros((T, K))

# Design matrix for cross-sectional regressions
X_cs = np.column_stack([np.ones(N), beta_hat])

for t in range(T):
    y = R[t, :]
    coef, _, _, _ = lstsq(X_cs, y, rcond=None)
    lambda_t[t] = coef[1:]  # skip intercept


Now lambda_t[t] is $(\lambda_{1,t}, \lambda_{2,t})$

### Step 3: Average prices of risk

In [5]:
lambda_bar = lambda_t.mean(axis=0)
lambda_bar

array([-0.07095088,  0.04180028])

This estimates $\bar{\lambda}_k = \frac{1}{T}\sum_{t=1}^{T} \lambda_{k,t} $

### Step 4: Fama-Macbeth t-statistics

In [6]:
# Sample variance of lambda_t
lambda_var = lambda_t.var(axis=0, ddof=1)

# Standard error of the mean
lambda_se = np.sqrt(lambda_var / T)

# t-statistics
t_stats = lambda_bar / lambda_se

pd.DataFrame({
    "lambda_bar": lambda_bar,
    "std_error": lambda_se,
    "t_stat": t_stats
}, index=[f"Factor {k+1}" for k in range(K)])


,lambda_bar,std_error,t_stat
Factor 1,-0.070951,0.082703,-0.857895
Factor 2,0.041800,0.096216,0.434442


### Step 5: Interpretation

In [7]:
print("True risk premia:", true_lambda)
print("Estimated risk premia:", lambda_bar)
print("t-statistics:", t_stats)

True risk premia: [ 0.5 -0.3]
Estimated risk premia: [-0.07095088  0.04180028]
t-statistics: [-0.85789531  0.43444156]
